<a href="https://colab.research.google.com/github/abdul2924/Machine_learning_Assignment/blob/main/Data_injection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Initial Setup and Project Structure
!apt-get update -qq > /dev/null
!apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
!pip install pyspark==3.5.0 pyarrow -q

import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"


from google.colab import drive
drive.mount('/content/drive', force_remount=True)


BASE_PATH = "/content/drive/MyDrive/yellow_tripdata_2019"
RAW_PATH = f"{BASE_PATH}/data/raw"
PROCESSED_PATH = f"{BASE_PATH}/data/processed/clean_taxi_data"


for folder in ["notebooks", "data/raw", "data/processed", "tableau", "config", "scripts", "tests"]:
    os.makedirs(os.path.join(BASE_PATH, folder), exist_ok=True)

print(f" Project mapped to: {BASE_PATH}")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 14.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.2 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.0 which is incompatible.
Mounted at /content/drive
 Project mapped to: /content/drive/MyDrive/yellow_tripdata_2019


In [ ]:
# Initialize Spark Session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYC_Taxi_2019_Ingestion") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "6g") \
    .config("spark.sql.parquet.mergeSchema", "false") \
    .getOrCreate()

print(" Spark Session Active")
print(f"Version: {spark.version}")

 Spark Session Active
Version: 3.5.0


In [ ]:
# Load Raw Data and Perform Initial Validation
from pyspark.sql.functions import col, isnan, when, count, date_format
from pyspark.sql.types import NumericType


print(f"Reading from: {RAW_PATH}")

df_raw = spark.read.parquet(f"{RAW_PATH}/yellow_2019_*.parquet")
total_rows = df_raw.count()
print(f"Total Raw Rows: {total_rows:,}")


check_cols = ["fare_amount", "trip_distance", "passenger_count", "tpep_pickup_datetime"]
null_exprs = []

for c in check_cols:

    if isinstance(df_raw.schema[c].dataType, NumericType):
        null_exprs.append(count(when(col(c).isNull() | isnan(col(c)), c)).alias(f"{c}_nulls"))
    else:
        null_exprs.append(count(when(col(c).isNull(), c)).alias(f"{c}_nulls"))

print("\n Null Value Summary ")
df_raw.select(*null_exprs).show()

print("\n Fare Amount Summary ")
df_raw.select("fare_amount").summary("count", "min", "mean", "max").show()

Reading from: /content/drive/MyDrive/yellow_tripdata_2019/data/raw
Total Raw Rows: 84,598,444

 Null Value Summary 
+-----------------+-------------------+---------------------+--------------------------+
|fare_amount_nulls|trip_distance_nulls|passenger_count_nulls|tpep_pickup_datetime_nulls|
+-----------------+-------------------+---------------------+--------------------------+
|                0|                  0|               444383|                         0|
+-----------------+-------------------+---------------------+--------------------------+


 Fare Amount Summary 
+-------+-----------------+
|summary|      fare_amount|
+-------+-----------------+
|  count|         84598444|
|    min|          -1856.0|
|   mean|13.41263973283576|
|    max|         943274.8|
+-------+-----------------+



In [ ]:
# List Contents of Raw Data Directory
import os


raw_data_dir = RAW_PATH

if os.path.exists(raw_data_dir):
    print(f"Contents of {raw_data_dir}:")
    for item in os.listdir(raw_data_dir):
        print(item)
else:
    print(f"The directory {raw_data_dir} does not exist.")

Contents of /content/drive/MyDrive/yellow_tripdata_2019/data/raw:
yellow_2019_01.parquet
yellow_2019_02.parquet
yellow_2019_03.parquet
yellow_2019_04.parquet
yellow_2019_05.parquet
yellow_2019_07.parquet
yellow_2019_06.parquet
yellow_2019_08.parquet
yellow_2019_09.parquet
yellow_2019_10.parquet
yellow_2019_11.parquet
yellow_2019_12.parquet


In [ ]:
# Clean and Filter Data
from pyspark.sql.functions import col, date_format, year


df_clean = df_raw.filter(
    (col("fare_amount").between(1.0, 500.0)) &
    (col("trip_distance").between(0.1, 100.0)) &
    (col("passenger_count").cast("int").between(1, 6)) &
    (year(col("tpep_pickup_datetime")) == 2019)
).dropDuplicates()


df_clean = df_clean.withColumn("pickup_month", date_format(col("tpep_pickup_datetime"), "yyyy-MM"))

print(f"Cleaned Data Rows: {df_clean.count():,}")

Cleaned Data Rows: 81,581,307


In [ ]:
# Write Cleaned Data to Processed Path
(df_clean.coalesce(50)
 .write
 .partitionBy("pickup_month")
 .mode("overwrite")
 .option("compression", "snappy")
 .parquet(PROCESSED_PATH))

print(f" Data successfully written to: {PROCESSED_PATH}")

 Data successfully written to: /content/drive/MyDrive/yellow_tripdata_2019/data/processed/clean_taxi_data


In [ ]:
# Verify Partition Distribution of Processed Data
df_final = spark.read.parquet(PROCESSED_PATH)

print("--- Partition Distribution ( only 2019 data) ")
df_final.groupBy("pickup_month").count().orderBy("pickup_month").show(12)

--- Partition Distribution ( only 2019 data) 
+------------+-------+
|pickup_month|  count|
+------------+-------+
|     2019-01|7476044|
|     2019-02|6830367|
|     2019-03|7616189|
|     2019-04|7223713|
|     2019-05|7340317|
|     2019-06|6724242|
|     2019-07|6068631|
|     2019-08|5835523|
|     2019-09|6315105|
|     2019-10|6930694|
|     2019-11|6603163|
|     2019-12|6617319|
+------------+-------+

